In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"
from functools import partial
import time
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy
import glob

import matplotlib as mpl
from matplotlib import rc
rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams.update({'font.size': 10})
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}\usepackage{amsmath}\usepackage{upgreek}"

In [ ]:
import jax
import jax.numpy as jnp
# jax.config.update("jax_enable_x64", True)
# jax.config.update("jax_debug_nans", True)
gpus = jax.devices()
print(gpus)

jax.config.update("jax_default_device", gpus[0])

import diffrax
import equinox as eqx
import optax

from haiku import PRNGSequence

from dmpe.data_management import DataPaths
from dmpe.evaluation.plotting_utils import plot_sequence
from dmpe.evaluation.experiment_utils import get_experiment_ids, load_experiment_results
from dmpe.models.models import NeuralEulerODECartpole
from dmpe.models.model_utils import simulate_ahead_with_env

In [ ]:
from dmpe.utils.env_utils.fluid_tank_utils import setup_env as setup_fluid_tank_env
from dmpe.utils.env_utils.pendulum_utils import setup_env as setup_pendulum_env
from dmpe.utils.env_utils.cart_pole_utils import setup_env as setup_cart_pole_env

In [ ]:
from dmpe.utils.density_estimation import build_grid
from dmpe.models.model_utils import simulate_ahead_with_env

In [ ]:
from dmpe.utils.sets.reachable_set import approximate_reachable_set
from dmpe.utils.sets.control_invariant_set import approximate_control_invariant_set

from dmpe.utils.sets.reachable_set import save_results as save_results_rs
from dmpe.utils.sets.control_invariant_set import save_results as save_results_ci
from dmpe.utils.sets.shared import load_results, DiscretizedSet, SlicedSet, load_discretized_set, save_discretized_set

In [ ]:
import matplotlib as mpl
from matplotlib import rc
rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams.update({'font.size': 10 * 2.54})
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}\usepackage{amsmath}\usepackage{upgreek}"

full_column_width = 18.2
half_column_width = 8.89

In [ ]:
from enum import Enum
class Systems(Enum):
    FLUID_TANK = 1
    PENDULUM = 2
    CART_POLE = 3

In [ ]:
S_xu_dict = {
    Systems.FLUID_TANK: load_discretized_set(DataPaths().reach_ci_experiments / "fluid_tank_S_xu_666a769d-8c1b-4b.json"),
    Systems.PENDULUM: load_discretized_set(DataPaths().reach_ci_experiments / "pendulum_S_xu_02430b86-ae0d-42.json"),
    Systems.CART_POLE: load_discretized_set(DataPaths().reach_ci_experiments / "cart_pole_S_xu_8744b5d5-30e6-4b.json"),
}

In [ ]:
S_xu_dict[Systems.CART_POLE]

## load $S^{(x,u)}$

In [ ]:
def visualize_S_xu(
    S_xu, reduction_method=jnp.mean, labels: None | list[str] = None, use_contourf: bool = True, scaling=1.0,
):
    dim = S_xu.grid.shape[-1]
    fig, axs = plt.subplots(nrows=dim, ncols=dim, figsize=(half_column_width*scaling, half_column_width*scaling), sharex=True, sharey=True)
    feature_indices = jnp.arange(0, dim, 1).tolist()

    if labels is None:
        labels = jnp.arange(0, dim, 1).tolist()

    for i in range(dim):
        for j in range(dim):

            axs[j, i].grid(True)

            reduction_indices = [f_idx for f_idx in feature_indices if not (f_idx == i or f_idx == j)]
            if len(reduction_indices) == dim - 1:
                continue

            reduced_safe = reduction_method(S_xu.mask_unflattened, axis=tuple(reduction_indices))

            if i > j:
                reduced_safe = jnp.transpose(reduced_safe)

            if use_contourf:
                contourset = axs[j, i].contourf(
                    S_xu.grid_unflattened[..., *[0 for _ in range(dim - 2)], 0],
                    S_xu.grid_unflattened[..., *[0 for _ in range(dim - 2)], 1],
                    reduced_safe,
                    vmin=0.0,
                    vmax=1.0,
                )
            else:
                axs[j, i].imshow(reduced_safe.T, origin="lower", extent=[-1, 1, -1, 1])
            axs[j, 0].set_ylabel(labels[j])
   
        axs[-1, i].set_xlabel(labels[i])
    #fig.tight_layout()
    return fig, axs

In [ ]:
for sys_name in Systems:

    if sys_name == Systems.FLUID_TANK:
        env, penalty_function, featurize, _ = setup_fluid_tank_env()
        S_xu = S_xu_dict[sys_name]
        labels = [r"$\tilde{h}$", r"$\tilde{q}_\mathrm{in}$"]
        scaling=0.65
    elif sys_name == Systems.PENDULUM:
        env, penalty_function, featurize, _ = setup_pendulum_env()
        S_xu = S_xu_dict[sys_name]
        labels = [r"$\tilde{\theta}$", r"$\tilde{\omega}$", r"$\tilde{T}$"]
        scaling=0.75
    elif sys_name == Systems.CART_POLE:
        env, penalty_function, featurize, _ = setup_cart_pole_env()
        S_xu = S_xu_dict[sys_name]
        labels = [r"$\tilde{d}$", r"$\tilde{v}$", r"$\tilde{\theta}$", r"$\tilde{\omega}$", r"$\tilde{F}$"]
        scaling=1.0

    fig, axs = visualize_S_xu(S_xu, use_contourf=True, labels=labels, scaling=scaling)
    plt.savefig(f"S_xu_visualization_{str(sys_name.name).lower()}.pdf", bbox_inches='tight');
    plt.show()

In [ ]:

# Create a colormap and a normalization
cmap = plt.cm.viridis
norm = mpl.colors.Normalize(vmin=0, vmax=1)

# Create a dummy ScalarMappable to use for the colorbar
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])  # Needed for some versions of matplotlib

# Create the figure and add only a colorbar
fig, ax = plt.subplots(figsize=(0.4, 0.65 * full_column_width))
fig.subplots_adjust(bottom=0.5)

cbar = fig.colorbar(sm, cax=ax, orientation='vertical')

plt.savefig("colorbar.svg", bbox_inches="tight")
plt.show()

In [ ]:
494.685 / 408.685 * 0.65